In [1]:
# ============================================================
# YouTube Comments Preprocessor
# K-pop Sentiment Analysis Project
# Filters English comments and cleans text
# ============================================================

import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from langdetect import detect, LangDetectException

nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# --- Load raw comments ---
df = pd.read_csv("../01_raw_data/youtube/youtube_comments_raw.csv")
print(f"Raw comments loaded: {len(df)}")
print(df.groupby("group_comeback").size())

# --- Filter English comments ---
def is_english(text):
    try:
        return detect(str(text)) == "en"
    except LangDetectException:
        return False

print("\nDetecting language...")
df["is_english"] = df["comment"].apply(is_english)
df_english = df[df["is_english"] == True].copy()
print(f"English comments: {len(df_english)} out of {len(df)}")
print(df_english.groupby("group_comeback").size())

# --- Clean text ---
stemmer = PorterStemmer()
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+|#\w+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess(text):
    cleaned = clean_text(text)
    tokens = word_tokenize(cleaned)
    tokens = [stemmer.stem(w) for w in tokens if w not in stop_words and len(w) > 2]
    return " ".join(tokens)

print("\nCleaning text...")
df_english["cleaned_comment"] = df_english["comment"].apply(clean_text)
df_english["processed_comment"] = df_english["comment"].apply(preprocess)
df_english = df_english[df_english["cleaned_comment"].str.len() > 5]
print(f"Comments after cleaning: {len(df_english)}")

# --- Save outputs ---
df_english.to_csv("../02_processed_data/youtube_comments_processed.csv", index=False)

labelling_df = df_english[["group_comeback", "comment", "cleaned_comment"]].copy()
labelling_df["sentiment"] = ""
labelling_df.to_csv("../03_labeled_data/youtube_comments_for_labelling.csv", index=False)

print("\nSaved:")
print("  02_processed_data/youtube_comments_processed.csv")
print("  03_labeled_data/youtube_comments_for_labelling.csv")
print("\nFinal breakdown:")
print(df_english.groupby("group_comeback").size())

Raw comments loaded: 3283
group_comeback
ATEEZ_IceOnMyTeeth        526
IVE_RebelHeart            576
NCTDREAM_WhenImWithYou    555
StrayKids_ChkChkBoom      538
TWICE_Strategy            523
aespa_Whiplash            565
dtype: int64

Detecting language...
English comments: 1096 out of 3283
group_comeback
ATEEZ_IceOnMyTeeth        280
IVE_RebelHeart            128
NCTDREAM_WhenImWithYou    184
StrayKids_ChkChkBoom      144
TWICE_Strategy            174
aespa_Whiplash            186
dtype: int64

Cleaning text...
Comments after cleaning: 1082

Saved:
  02_processed_data/youtube_comments_processed.csv
  03_labeled_data/youtube_comments_for_labelling.csv

Final breakdown:
group_comeback
ATEEZ_IceOnMyTeeth        278
IVE_RebelHeart            126
NCTDREAM_WhenImWithYou    184
StrayKids_ChkChkBoom      141
TWICE_Strategy            170
aespa_Whiplash            183
dtype: int64


In [1]:
import pandas as pd

df = pd.read_csv("../03_labeled_data/youtube_comments_for_labelling.csv")
print(f"Before deduplication: {len(df)}")

df = df.drop_duplicates(subset=["comment"], keep="first")
print(f"After deduplication: {len(df)}")
print(df.groupby("group_comeback").size())

df.to_csv("../03_labeled_data/youtube_comments_for_labelling.csv", index=False)
print("Saved.")

Before deduplication: 1082
After deduplication: 1038
group_comeback
ATEEZ_IceOnMyTeeth        273
IVE_RebelHeart            109
NCTDREAM_WhenImWithYou    180
StrayKids_ChkChkBoom      140
TWICE_Strategy            164
aespa_Whiplash            172
dtype: int64
Saved.
